# Week 02｜地震地図をつくる・改良する

授業の最後の30分に始め、このNotebookを宿題として仕上げます。最初は**CSVの一行につき、マーカーを一つ**。そこから自分の地図へ。


## 1｜CSVを用意する

授業サイトから**`eq.csv`**をダウンロードし、このNotebookと同じフォルダに保存します。2026年10月7日時点の地震データです。

もっと新しいデータを使うなら、[USGSのCSVダウンロードページ](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php)で **Past Day → All Earthquakes** を選びます。ダウンロードしたCSVの名前に`csv_file`を変えてください。

最初の一度だけ、下のCellで地図の道具`folium`をインストールします。CSVを読む`csv`はPythonに最初から入っています。


In [ ]:
%pip install folium

## 2｜CSVを読む

`latitude`は緯度、`longitude`は経度、`mag`はマグニチュード。CSVの値は文字なので、地図に使う緯度・経度は`float()`で数に変えます。


In [ ]:
import csv
import folium

csv_file = "eq.csv"

with open(csv_file, encoding="utf-8-sig") as file:
    earthquakes = list(csv.DictReader(file))

print(f"{len(earthquakes)}件のデータを読みました")

## 3｜一行につき、マーカーを一つ

実行する前に、どこにどんな点が出るか予想しよう。`for`の中が各地震に対して繰り返されます。最後の`earthquake_map`でNotebookに地図を表示します。


In [ ]:
earthquake_map = folium.Map(location=[20, 0], zoom_start=2)

for quake in earthquakes:
    folium.Marker(
        location=[float(quake["latitude"]), float(quake["longitude"])],
        tooltip=f"M {quake['mag']}"
    ).add_to(earthquake_map)

earthquake_map

## Challenge｜この地図を改良しよう

授業では一つ改良を始めます。宿題では**二つ以上の改良**を完成させよう。下のコードCellに、上の地図のコードをコピーして変更します。

| 改良の例 | ヒント |
| --- | --- |
| ベースマップを変える | `folium.Map(..., tiles="CartoDB positron")`。地図の見え方を比べる。 |
| ピンを円にする | `folium.Marker`を`folium.CircleMarker`に変え、`radius=5`、`fill=True`を追加する。 |
| 大きさによって色を変える | `float(quake["mag"])`と`if`で色を選び、円の`color`と`fill_color`に使う。空欄の`mag`は先に除く。 |
| 大きな地震だけ表示する | `if quake["mag"] and float(quake["mag"]) >= 4:`の中でマーカーを作る。 |
| 自分の工夫をする | 地域を絞る、円の大きさを変える、クリックで情報を見せる、など。 |

どの改良も、変更する前に結果を予想してから実行しよう。フィルターを使ったら、条件を満たす地震だけ表示されているか確認する。


In [2]:
# 上の地図のコードをコピーして、自分の改良を加えよう。
import csv
import json
from pathlib import Path
from IPython.display import IFrame, display

# CSVファイルを読み込む
csv_file = Path("eq.csv")

if not csv_file.is_file():
    raise FileNotFoundError(
        "eq.csvが見つかりません。Notebookと同じフォルダーに置いてください。"
    )

with open(csv_file, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

# 必要なデータを取り出す
earthquakes = []

for row in rows:
    try:
        lat = float(row["latitude"])
        lon = float(row["longitude"])
        mag = float(row["mag"])

        if -90 <= lat <= 90 and -180 <= lon <= 180:
            earthquakes.append({
                "lat": lat,
                "lon": lon,
                "mag": mag
            })
    except (ValueError, KeyError, TypeError):
        continue

if not earthquakes:
    raise ValueError(
        "地震データがありません。CSVの列名が latitude、longitude、mag か確認してください。"
    )

# 地図の中心を計算する
center_lat = sum(q["lat"] for q in earthquakes) / len(earthquakes)
center_lon = sum(q["lon"] for q in earthquakes) / len(earthquakes)

# 地震データをJSONに変換する
earthquake_json = json.dumps(earthquakes)

# HTML地図を作成する
html = """
<!DOCTYPE html>
<html lang="ja">
<head>
<meta charset="utf-8">
<title>世界の地震マップ</title>
<link rel="stylesheet"
 href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>
body { margin: 0; font-family: sans-serif; }
h2 { padding: 10px; }
#map { height: 600px; }
</style>
</head>
<body>
<h2>世界の地震マップ（マグニチュード別）</h2>
<div id="map"></div>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
const earthquakes = EARTHQUAKE_DATA;

const map = L.map("map").setView(
    [CENTER_LAT, CENTER_LON], 2
);

// 改良1：ベースマップを変更する
L.tileLayer(
    "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    {
        attribution: "&copy; OpenStreetMap contributors &copy; CARTO"
    }
).addTo(map);

// 改良2：ピンを円に変更する
// 改良3：マグニチュードによって色と大きさを変える
earthquakes.forEach(function(q) {
    let color = "blue";
    let radius = 3;

    if (q.mag >= 6) {
        color = "red";
        radius = 9;
    } else if (q.mag >= 5) {
        color = "orange";
        radius = 7;
    } else if (q.mag >= 4) {
        color = "green";
        radius = 5;
    }

    L.circleMarker([q.lat, q.lon], {
        radius: radius,
        color: color,
        fillColor: color,
        fillOpacity: 0.65
    })
    .bindPopup("マグニチュード：" + q.mag.toFixed(1))
    .addTo(map);
});
</script>
</body>
</html>
"""

# HTML内のデータを置き換える
html = html.replace("EARTHQUAKE_DATA", earthquake_json)
html = html.replace("CENTER_LAT", str(center_lat))
html = html.replace("CENTER_LON", str(center_lon))

# HTMLファイルを作成する
output_file = Path.cwd() / "earthquake_map.html"
output_file.write_text(html, encoding="utf-8")

print("地震データの読み込み件数：", len(earthquakes))
print("地図ファイルを作成しました：", output_file)

# Notebook内に地図を表示する
display(IFrame(
    src=output_file.resolve().as_uri(),
    width="100%",
    height=620
))

地震データの読み込み件数： 1850
地図ファイルを作成しました： [local]\Downloads\earthquake_map.html


## 宿題｜自分の地震地図

授業で始めた地図を仕上げます。別のNotebookを作る必要はありません。

- タイトルと、誰に見せる地図かを書く。
- 二つ以上改良し、その理由と地図から気づいたことを書く。
- CSVの出典とダウンロード日を書く（付属CSVなら2026年10月7日時点のデータと明記）。
- 上から全Cellを実行し、地図が表示された状態で保存する。
- **完成した`.ipynb`と使用したCSVをUNIPAに提出する。**

**締切：2026年10月14日(水) 23:59 JST**


### 私の地図

**タイトル：**
世界の地震をマグニチュード別に見る地図

**誰に見せたい？：**
地震や防災に関心がある学生や、世界のどこで地震が起きているのかを知りたい人に見せたい。

**改良したこと・理由：**
1つ目は、地図の背景を変更したことです。背景をシンプルにすることで、地震が発生した場所を見つけやすくしました。2つ目は、ピンを円に変更したことです。3つ目は、マグニチュードによって円の色と大きさを変えたことです。規模の大きい地震ほど目立つようにすることで、地震の違いを比較しやすくしました。また、円をクリックするとマグニチュードが表示されるようにしました。

**地図から気づいたこと：**
地図を見ると、地震は世界中で均等に起きているのではなく、特定の地域に集まっていることが分かりました。また、円の色や大きさを変えることで、地震の規模の違いが分かりやすくなりました。地震が多く発生する地域を知ることは、防災について考えるきっかけにもなると思いました。

**確かめたこと：**
地図の背景が変更されていること、地震がピンではなく円で表示されていること、マグニチュードによって円の色と大きさが変化することを確認しました。また、円をクリックするとマグニチュードが表示されることも確認しました。

**CSVの出典・ダウンロード日：**
出典：授業で配布された `eq.csv`  

